# 11 — DNN regressor for Sérsic parameters

Train a CNN+MLP to map a galaxy image directly to its 7-D Sérsic parameter vector. At inference time this replaces a few thousand Adam iterations with a single forward pass — a useful technique for very large surveys.

In [ ]:
# Bootstrap: make `lensing` importable when running notebooks/ directly.
import sys
from pathlib import Path
repo = Path.cwd().resolve().parent
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

import lensing as gl
# Device-agnostic: prefer MPS (Apple GPU) → CUDA → CPU.
# Pass device="cpu" if you need to force the CPU path (e.g. for
# operators that have no MPS kernel yet, or for reproducibility).
device, dtype = gl.config.setup(seed=42)
print(f"using device: {device}")


In [ ]:
from torch.utils.data import DataLoader

train = gl.ml.datasets.SersicParamDataset(n_samples=1500, npix=48,
                                           deltapix=0.05, seed=0)
val   = gl.ml.datasets.SersicParamDataset(n_samples=200, npix=48,
                                           deltapix=0.05, seed=10000)
train_loader = DataLoader(train, batch_size=32, shuffle=True)
val_loader   = DataLoader(val,   batch_size=32)

model = gl.ml.models.SersicRegressor(in_channels=1, n_outputs=7)
history = gl.ml.train.fit_model(
    model, train_loader, val_loader,
    loss_fn=nn.MSELoss(),
    lr=1e-3, epochs=10,
    metrics={'mse': gl.ml.train.mse},
    log_every=1,
)


## Per-parameter accuracy

In [ ]:
from lensing.ml.datasets import PARAM_KEYS

test = gl.ml.datasets.SersicParamDataset(n_samples=300, npix=48,
                                          deltapix=0.05, seed=98765)
preds, truths = [], []
model.eval()
with torch.no_grad():
    for x, y in DataLoader(test, batch_size=64):
        # Move to the trained model's device, then back to CPU
        # for downstream NumPy / Matplotlib code.
        preds.append(model(x.to(device)).cpu().numpy())
        truths.append(y.numpy())
preds = np.vstack(preds); truths = np.vstack(truths)

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for i, (ax, name) in enumerate(zip(axes.flatten(), PARAM_KEYS)):
    ax.scatter(truths[:, i], preds[:, i], s=8, alpha=0.5)
    lo, hi = truths[:, i].min(), truths[:, i].max()
    ax.plot([lo, hi], [lo, hi], 'k--')
    ax.set(xlabel=f'true {name}', ylabel=f'pred {name}',
           title=f'r = {np.corrcoef(truths[:,i], preds[:,i])[0,1]:.3f}')
axes[1, 3].axis('off')
plt.tight_layout(); plt.show()


## Comparison vs. classical Adam fit

How much does inference speed up if we use the DNN as a one-shot estimator vs. running Adam from scratch on each image?

In [ ]:
import time

x_one, y_true = test[0]
t0 = time.perf_counter()
with torch.no_grad():
    y_dnn = model(x_one.unsqueeze(0).to(device))[0].cpu().numpy()
t_dnn = time.perf_counter() - t0
print(f'DNN inference: {t_dnn*1e3:.2f} ms')

# Adam fit baseline
xy = train._xy
t0 = time.perf_counter()
galaxy = gl.light.Sersic(Ie=1., Re=1., n=2.5, x0=0., y0=0., e1=0., e2=0.)
res = gl.inference.fit(
    galaxy, xy, x_one[0],
    gl.inference.ReducedChiSquared(sigma=0.05, n_params=7),
    lr=0.05, epochs=500,
)
t_adam = time.perf_counter() - t0
print(f'Adam fit (500 epochs): {t_adam*1e3:.0f} ms — speedup ~ {t_adam/t_dnn:.0f}x')
